In [ ]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


print("Upload your CLASS A CSV:")
uploaded_a = files.upload()

class_a_file = list(uploaded_a.keys())[0]

class_a = pd.read_csv(class_a_file)

print("\nClass A columns:")
print(class_a.columns.tolist())


print("\nUpload your CLASS B CSV:")
uploaded_b = files.upload()

class_b_file = list(uploaded_b.keys())[0]

class_b = pd.read_csv(class_b_file)

print("\nClass B columns:")
print(class_b.columns.tolist())

Upload your CLASS A CSV:


Saving Class_A_Criminal_Usernames_Master_Database.csv to Class_A_Criminal_Usernames_Master_Database.csv

Class A columns:
['Real Name', 'Username', 'Age', 'Sex', 'Nationality', 'Crime', 'Source', 'Source URL', 'Review Status', 'Notes']

Upload your CLASS B CSV:


Saving Class_B_X_YouTube_Final.csv to Class_B_X_YouTube_Final.csv

Class B columns:
['item', 'qid', 'platform', 'property_id', 'handle', 'itemLabel', 'occupations', 'gender', 'citizenships', 'languages', 'date_of_birth', 'age']


In [ ]:
def find_username_column(df):
    possible_names = [
        "username",
        "user_name",
        "handle",
        "user",
        "name"
    ]

    columns_lower = {
        str(col).lower().strip(): col
        for col in df.columns
    }

    for name in possible_names:
        if name in columns_lower:
            return columns_lower[name]

    raise ValueError(
        f"Could not automatically find username column. "
        f"Columns found: {df.columns.tolist()}"
    )


a_username_col = find_username_column(class_a)
b_username_col = find_username_column(class_b)

print("Class A username column:", a_username_col)
print("Class B username column:", b_username_col)

Class A username column: Username
Class B username column: handle


In [ ]:
class_a_clean = class_a[[a_username_col]].copy()
class_a_clean.columns = ["username"]

class_b_clean = class_b[[b_username_col]].copy()
class_b_clean.columns = ["username"]


class_a_clean = class_a_clean.dropna()
class_b_clean = class_b_clean.dropna()


class_a_clean["username"] = class_a_clean["username"].astype(str)
class_b_clean["username"] = class_b_clean["username"].astype(str)


class_a_clean = class_a_clean.drop_duplicates("username")
class_b_clean = class_b_clean.drop_duplicates("username")



class_a_clean["label"] = 1
class_b_clean["label"] = 0


print("Class A usernames:", len(class_a_clean))
print("Class B usernames:", len(class_b_clean))

Class A usernames: 234
Class B usernames: 19750


In [ ]:
n = min(len(class_a_clean), len(class_b_clean))

class_a_sample = class_a_clean.sample(
    n=n,
    random_state=42
)

class_b_sample = class_b_clean.sample(
    n=n,
    random_state=42
)


data = pd.concat(
    [class_a_sample, class_b_sample],
    ignore_index=True
)

data = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print(data["label"].value_counts())

data.head()

label
1    234
0    234
Name: count, dtype: int64


,username,label
0,Yimmy702,1
1,BadB,1
2,Lux,1
3,simmjazeera,0
4,Bx1,1


#Testing Section

In [ ]:
X = data["username"]
y = data["label"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Training usernames:", len(X_train))
print("Testing usernames:", len(X_test))

Training usernames: 374
Testing usernames: 94


In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True
)


X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)


print("Number of features:", X_train_tfidf.shape[1])

Number of features: 6991


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")

Model trained successfully.


In [ ]:
predictions = model.predict(X_test_tfidf)


print("Accuracy:")
print(accuracy_score(y_test, predictions))


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=[
            "Class B",
            "Class A"
        ]
    )
)


print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

Accuracy:
0.6276595744680851

Classification Report:
              precision    recall  f1-score   support

     Class B       0.60      0.79      0.68        47
     Class A       0.69      0.47      0.56        47

    accuracy                           0.63        94
   macro avg       0.64      0.63      0.62        94
weighted avg       0.64      0.63      0.62        94


Confusion Matrix:
[[37 10]
 [25 22]]


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

import numpy as np

X = data["username"]
y = data["label"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
pr_auc_scores = []


fold = 1

for train_index, test_index in skf.split(X, y):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        lowercase=True
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)


    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train_tfidf,
        y_train
    )

    predictions = model.predict(
        X_test_tfidf
    )

    probabilities = model.predict_proba(
        X_test_tfidf
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions
    )

    recall = recall_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )


    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    pr_auc_scores.append(pr_auc)

    print(f"\nFOLD {fold}")
    print("--------------------")

    print(
        "Accuracy:",
        round(accuracy, 3)
    )

    print(
        "Class A Precision:",
        round(precision, 3)
    )

    print(
        "Class A Recall:",
        round(recall, 3)
    )

    print(
        "Class A F1:",
        round(f1, 3)
    )

    print(
        "PR-AUC:",
        round(pr_auc, 3)
    )


    fold += 1

print("5-FOLD CROSS-VALIDATION RESULTS")
print("CLASS A vs CLASS B")


print(
    "Accuracy:",
    round(np.mean(accuracy_scores), 3),
    "+/-",
    round(np.std(accuracy_scores), 3)
)


print(
    "Class A Precision:",
    round(np.mean(precision_scores), 3),
    "+/-",
    round(np.std(precision_scores), 3)
)


print(
    "Class A Recall:",
    round(np.mean(recall_scores), 3),
    "+/-",
    round(np.std(recall_scores), 3)
)


print(
    "Class A F1:",
    round(np.mean(f1_scores), 3),
    "+/-",
    round(np.std(f1_scores), 3)
)


print(
    "PR-AUC:",
    round(np.mean(pr_auc_scores), 3),
    "+/-",
    round(np.std(pr_auc_scores), 3)
)


FOLD 1
--------------------
Accuracy: 0.798
Class A Precision: 0.804
Class A Recall: 0.787
Class A F1: 0.796
PR-AUC: 0.842

FOLD 2
--------------------
Accuracy: 0.617
Class A Precision: 0.617
Class A Recall: 0.617
Class A F1: 0.617
PR-AUC: 0.691

FOLD 3
--------------------
Accuracy: 0.617
Class A Precision: 0.657
Class A Recall: 0.489
Class A F1: 0.561
PR-AUC: 0.749

FOLD 4
--------------------
Accuracy: 0.71
Class A Precision: 0.794
Class A Recall: 0.574
Class A F1: 0.667
PR-AUC: 0.773

FOLD 5
--------------------
Accuracy: 0.677
Class A Precision: 0.722
Class A Recall: 0.565
Class A F1: 0.634
PR-AUC: 0.789

5-FOLD CROSS-VALIDATION RESULTS
CLASS A vs CLASS B
Accuracy: 0.684 +/- 0.067
Class A Precision: 0.719 +/- 0.074
Class A Recall: 0.607 +/- 0.099
Class A F1: 0.655 +/- 0.078
PR-AUC: 0.769 +/- 0.05


In [ ]:
def classify_username(username):

    username_vector = vectorizer.transform([username])

    prediction = model.predict(username_vector)[0]

    probability = model.predict_proba(username_vector)[0][1]

    if prediction == 1:
        result = "Class A-like"
    else:
        result = "Class B-like"

    print("Username:", username)
    print("Prediction:", result)
    print(
        "Class A probability:",
        round(probability, 3)
    )

In [ ]:
print(classify_username('Craig_Levine'))

Username: Craig_Levine
Prediction: Class B-like
Class A probability: 0.472
None
